In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.ticker import PercentFormatter
import plotly.graph_objects as go
from scipy.stats import fisher_exact
import numpy as np

In [19]:
# Skip the first 11 rows
df = pd.read_csv(r"C:\Users\CastroJG\Downloads\Unsignalized_Intersections_3-17-26_v2(in).csv", skiprows=22)

# 2️⃣ Rename columns first: remove spaces and lowercase
df.columns = df.columns.str.strip().str.replace(' ', '_').str.lower()

# 3️⃣ Replace any 'No Data' strings with NaN
df.replace({"latitude": {"No Data": np.nan},
            "longitude": {"No Data": np.nan}}, inplace=True)

# 4️⃣ Drop rows where critical columns are NaN
critical_cols = ["latitude", "longitude"]
df = df.dropna(subset=critical_cols)

# 5️⃣ Convert Latitude and Longitude to float (double precision)
df["latitude"] = df["latitude"].astype(np.float64)
df["longitude"] = df["longitude"].astype(np.float64)

df

,crash_id,city,crash_date,crash_month,crash_severity,crash_time,crash_year,intersecting_street_name,latitude,light_condition,...,speed_limit,street_name,surface_condition,weather_condition,contributing_factor_1,contributing_factor_2,contributing_factor_3,possible_contributing_factor_1,possible_contributing_factor_2,person_type
0,17499191,EL PASO,1/1/2020,1,N - NOT INJURED,346,2020,E MISSOURI AVE,31.762815,"3 - DARK, LIGHTED",...,35,N KANSAS ST,1 - DRY,2 - CLOUDY,20 - DRIVER INATTENTION,No Data,No Data,No Data,No Data,1 - DRIVER
1,17499191,EL PASO,1/1/2020,1,N - NOT INJURED,346,2020,E MISSOURI AVE,31.762815,"3 - DARK, LIGHTED",...,35,N KANSAS ST,1 - DRY,2 - CLOUDY,20 - DRIVER INATTENTION,No Data,No Data,No Data,No Data,2 - PASSENGER/OCCUPANT
2,17499191,EL PASO,1/1/2020,1,N - NOT INJURED,346,2020,E MISSOURI AVE,31.762815,"3 - DARK, LIGHTED",...,35,N KANSAS ST,1 - DRY,2 - CLOUDY,20 - DRIVER INATTENTION,No Data,No Data,No Data,No Data,2 - PASSENGER/OCCUPANT
3,17499191,EL PASO,1/1/2020,1,N - NOT INJURED,346,2020,E MISSOURI AVE,31.762815,"3 - DARK, LIGHTED",...,35,N KANSAS ST,1 - DRY,2 - CLOUDY,20 - DRIVER INATTENTION,No Data,No Data,No Data,No Data,1 - DRIVER
4,17499191,EL PASO,1/1/2020,1,N - NOT INJURED,346,2020,E MISSOURI AVE,31.762815,"3 - DARK, LIGHTED",...,35,N KANSAS ST,1 - DRY,2 - CLOUDY,20 - DRIVER INATTENTION,No Data,No Data,No Data,No Data,2 - PASSENGER/OCCUPANT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10862,21215102,EL PASO,12/27/2025,12,N - NOT INJURED,1517,2025,S MESA HILLS DR,31.819401,1 - DAYLIGHT,...,40,SUNLAND PARK DR,1 - DRY,1 - CLEAR,20 - DRIVER INATTENTION,No Data,No Data,No Data,No Data,1 - DRIVER
10863,21215102,EL PASO,12/27/2025,12,N - NOT INJURED,1517,2025,S MESA HILLS DR,31.819401,1 - DAYLIGHT,...,40,SUNLAND PARK DR,1 - DRY,1 - CLEAR,20 - DRIVER INATTENTION,No Data,No Data,No Data,No Data,2 - PASSENGER/OCCUPANT
10864,21207059,EL PASO,12/29/2025,12,C - POSSIBLE INJURY,835,2025,RICHMOND AVE,31.795915,"3 - DARK, LIGHTED",...,30,ALABAMA ST,2 - WET,3 - RAIN,22 - FAILED TO CONTROL SPEED,No Data,No Data,20 - DRIVER INATTENTION,No Data,1 - DRIVER
10865,21207059,EL PASO,12/29/2025,12,C - POSSIBLE INJURY,835,2025,RICHMOND AVE,31.795915,"3 - DARK, LIGHTED",...,30,ALABAMA ST,2 - WET,3 - RAIN,No Data,No Data,No Data,No Data,No Data,1 - DRIVER


In [20]:
severity_values = {
    'K - FATAL INJURY': 13749500,
    'A - SUSPECTED SERIOUS INJURY': 1432400,
    'B - SUSPECTED MINOR INJURY': 303200,
    'C - POSSIBLE INJURY': 151600,
    'N - NOT INJURED': 0,
    '99 - UNKNOWN': 0
}

df["Collision KABCO USD"] = df["crash_severity"].map(severity_values).fillna(0)

In [21]:
%%sql
WITH base AS (
    SELECT
        crash_id,

        -- Canonical intersection (handles reversed streets)
        CONCAT(
            LEAST(street_name, intersecting_street_name),
            ' & ',
            GREATEST(street_name, intersecting_street_name)
        ) AS intersection,

        latitude,
        longitude,
        "Collision KABCO USD" AS kabco,

        contributing_factor_1,
        contributing_factor_2,
        contributing_factor_3,
        possible_contributing_factor_1,
        possible_contributing_factor_2,

        person_type,
        manner_of_collision

    FROM df
    WHERE street_name IS NOT NULL
      AND intersecting_street_name IS NOT NULL
), intersection_summary AS (
    SELECT
        intersection,
        AVG(latitude) AS avg_latitude,
        AVG(longitude) AS avg_longitude,
        SUM(kabco) AS total_kabco,

        RANK() OVER (ORDER BY SUM(kabco) DESC) AS rank
    FROM base
    GROUP BY intersection
), contributing_long AS (
    SELECT intersection, contributing_factor_1 AS factor FROM base
    UNION ALL
    SELECT intersection, contributing_factor_2 FROM base
    UNION ALL
    SELECT intersection, contributing_factor_3 FROM base
    UNION ALL
    SELECT intersection, possible_contributing_factor_1 FROM base
    UNION ALL
    SELECT intersection, possible_contributing_factor_2 FROM base
),

contributing_pivot AS (
    SELECT
        intersection,

        SUM(CASE WHEN factor = 'Driver Inattention' THEN 1 ELSE 0 END) AS driver_inattention,
        SUM(CASE WHEN factor = 'Speeding' THEN 1 ELSE 0 END) AS speeding,
        SUM(CASE WHEN factor = 'Failure to Yield' THEN 1 ELSE 0 END) AS failure_to_yield

        -- 🔥 Add more factors as needed

    FROM contributing_long
    WHERE factor IS NOT NULL
    GROUP BY intersection
), person_type_pivot AS (
    SELECT
        intersection,

        SUM(CASE WHEN person_type = 'Driver' THEN 1 ELSE 0 END) AS driver,
        SUM(CASE WHEN person_type = 'Pedestrian' THEN 1 ELSE 0 END) AS pedestrian,
        SUM(CASE WHEN person_type = 'Motorcyclist' THEN 1 ELSE 0 END) AS motorcyclist

    FROM base
    GROUP BY intersection
), collision_pivot AS (
    SELECT
        intersection,

        SUM(CASE WHEN manner_of_collision = 'Rear-end' THEN 1 ELSE 0 END) AS rear_end,
        SUM(CASE WHEN manner_of_collision = 'Angle' THEN 1 ELSE 0 END) AS angle,
        SUM(CASE WHEN manner_of_collision = 'Sideswipe' THEN 1 ELSE 0 END) AS sideswipe

    FROM base
    GROUP BY intersection
)
SELECT
    s.intersection,
    s.avg_latitude,
    s.avg_longitude,
    s.total_kabco,
    s.rank,

    cp.*,
    pt.*,
    mc.*

FROM intersection_summary s
LEFT JOIN contributing_pivot cp ON s.intersection = cp.intersection
LEFT JOIN person_type_pivot pt ON s.intersection = pt.intersection
LEFT JOIN collision_pivot mc ON s.intersection = mc.intersection

ORDER BY s.rank;

,intersection,avg_latitude,avg_longitude,total_kabco,rank,intersection_1,driver_inattention,speeding,failure_to_yield,intersection_2,driver,pedestrian,motorcyclist,intersection_3,rear_end,angle,sideswipe
0,HOWARD ST & TITANIC AVE,31.853444,-106.433330,152154100.0,1,HOWARD ST & TITANIC AVE,0.0,0.0,0.0,HOWARD ST & TITANIC AVE,0.0,0.0,0.0,HOWARD ST & TITANIC AVE,0.0,0.0,0.0
1,PEBBLE HILLS BLVD & TIERRA BLANDA DR,31.777093,-106.260784,68747500.0,2,PEBBLE HILLS BLVD & TIERRA BLANDA DR,0.0,0.0,0.0,PEBBLE HILLS BLVD & TIERRA BLANDA DR,0.0,0.0,0.0,PEBBLE HILLS BLVD & TIERRA BLANDA DR,0.0,0.0,0.0
2,DIANA DR & TETONS DR,31.858941,-106.423022,43370900.0,3,DIANA DR & TETONS DR,0.0,0.0,0.0,DIANA DR & TETONS DR,0.0,0.0,0.0,DIANA DR & TETONS DR,0.0,0.0,0.0
3,N PIEDRAS ST & N PIEDRAS ST,31.813975,-106.461320,41248500.0,4,N PIEDRAS ST & N PIEDRAS ST,0.0,0.0,0.0,N PIEDRAS ST & N PIEDRAS ST,0.0,0.0,0.0,N PIEDRAS ST & N PIEDRAS ST,0.0,0.0,0.0
4,ENCHANTED BREEZE DR & ENCHANTED PATH DR,31.918735,-106.572240,13749500.0,5,ENCHANTED BREEZE DR & ENCHANTED PATH DR,0.0,0.0,0.0,ENCHANTED BREEZE DR & ENCHANTED PATH DR,0.0,0.0,0.0,ENCHANTED BREEZE DR & ENCHANTED PATH DR,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1918,BARTLETT DR & N MESA ST,31.838850,-106.566979,0.0,834,BARTLETT DR & N MESA ST,0.0,0.0,0.0,BARTLETT DR & N MESA ST,0.0,0.0,0.0,BARTLETT DR & N MESA ST,0.0,0.0,0.0
1919,COYOTE TRAIL DR & TIM FOSTER ST,31.787049,-106.225160,0.0,834,COYOTE TRAIL DR & TIM FOSTER ST,0.0,0.0,0.0,COYOTE TRAIL DR & TIM FOSTER ST,0.0,0.0,0.0,COYOTE TRAIL DR & TIM FOSTER ST,0.0,0.0,0.0
1920,DELTA DR & TROWBRIDGE DR,31.763431,-106.395927,0.0,834,DELTA DR & TROWBRIDGE DR,0.0,0.0,0.0,DELTA DR & TROWBRIDGE DR,0.0,0.0,0.0,DELTA DR & TROWBRIDGE DR,0.0,0.0,0.0
1921,PARTELLO ST & PORTER AVE,31.804915,-106.446230,0.0,834,PARTELLO ST & PORTER AVE,0.0,0.0,0.0,PARTELLO ST & PORTER AVE,0.0,0.0,0.0,PARTELLO ST & PORTER AVE,0.0,0.0,0.0


In [27]:
%%sql
WITH base AS (
    SELECT
        crash_id,

        CONCAT(
            LEAST(street_name, intersecting_street_name),
            ' & ',
            GREATEST(street_name, intersecting_street_name)
        ) AS intersection,

        latitude,
        longitude,
        "Collision KABCO USD" AS kabco,

        contributing_factor_1,
        contributing_factor_2,
        contributing_factor_3,
        possible_contributing_factor_1,
        possible_contributing_factor_2,

        person_type,
        manner_of_collision

    FROM df
    WHERE street_name IS NOT NULL
      AND intersecting_street_name IS NOT NULL
      AND latitude IS NOT NULL
      AND longitude IS NOT NULL
),

intersection_summary AS (
    SELECT
        intersection,
        AVG(latitude) AS avg_latitude,
        AVG(longitude) AS avg_longitude,
        SUM(kabco) AS total_kabco,
        RANK() OVER (ORDER BY SUM(kabco) DESC) AS rank
    FROM base
    GROUP BY intersection
), contributing_long AS (
    SELECT intersection, contributing_factor_1 AS factor FROM base
    UNION ALL
    SELECT intersection, contributing_factor_2 FROM base
    UNION ALL
    SELECT intersection, contributing_factor_3 FROM base
    UNION ALL
    SELECT intersection, possible_contributing_factor_1 FROM base
    UNION ALL
    SELECT intersection, possible_contributing_factor_2 FROM base
)

SELECT
    s.intersection,
    s.avg_latitude,
    s.avg_longitude,
    s.total_kabco,
    s.rank,

    cl.factor,
    COUNT(*) AS factor_count

FROM intersection_summary s
LEFT JOIN contributing_long cl
    ON s.intersection = cl.intersection
WHERE cl.factor IS NOT NULL
GROUP BY
    s.intersection,
    s.avg_latitude,
    s.avg_longitude,
    s.total_kabco,
    s.rank,
    cl.factor
ORDER BY s.rank, factor_count DESC;

,intersection,avg_latitude,avg_longitude,total_kabco,rank,factor,factor_count
0,HOWARD ST & TITANIC AVE,31.853444,-106.43333,152154100.0,1,No Data,53
1,HOWARD ST & TITANIC AVE,31.853444,-106.43333,152154100.0,1,20 - DRIVER INATTENTION,8
2,HOWARD ST & TITANIC AVE,31.853444,-106.43333,152154100.0,1,45 - HAD BEEN DRINKING,7
3,HOWARD ST & TITANIC AVE,31.853444,-106.43333,152154100.0,1,60 - UNSAFE SPEED,4
4,HOWARD ST & TITANIC AVE,31.853444,-106.43333,152154100.0,1,16 - DISREGARD STOP SIGN OR LIGHT,1
...,...,...,...,...,...,...,...
6321,BARBADOS CT & SAUL KLEINFELD DR,31.800005,-106.28533,0.0,834,20 - DRIVER INATTENTION,1
6322,FRESNO DR & S YARBROUGH DR,31.719795,-106.35942,0.0,834,20 - DRIVER INATTENTION,1
6323,BLANCHARD AVE & N KANSAS ST,31.773705,-106.49760,0.0,834,20 - DRIVER INATTENTION,1
6324,E CALIFORNIA AVE & N CAMPBELL ST,31.768465,-106.49163,0.0,834,98 - OTHER (EXPLAIN IN NARRATIVE),1


In [28]:
%%sql
WITH base AS (
    SELECT
        crash_id,

        CONCAT(
            LEAST(street_name, intersecting_street_name),
            ' & ',
            GREATEST(street_name, intersecting_street_name)
        ) AS intersection,

        latitude,
        longitude,
        "Collision KABCO USD" AS kabco,

        contributing_factor_1,
        contributing_factor_2,
        contributing_factor_3,
        possible_contributing_factor_1,
        possible_contributing_factor_2,

        person_type,
        manner_of_collision

    FROM df
    WHERE street_name IS NOT NULL
      AND intersecting_street_name IS NOT NULL
      AND latitude IS NOT NULL
      AND longitude IS NOT NULL
),

intersection_summary AS (
    SELECT
        intersection,
        AVG(latitude) AS avg_latitude,
        AVG(longitude) AS avg_longitude,
        SUM(kabco) AS total_kabco,
        RANK() OVER (ORDER BY SUM(kabco) DESC) AS rank
    FROM base
    GROUP BY intersection
)SELECT
    s.intersection,
    s.avg_latitude,
    s.avg_longitude,
    s.total_kabco,
    s.rank,

    b.person_type,
    COUNT(*) AS person_type_count

FROM intersection_summary s
LEFT JOIN base b
    ON s.intersection = b.intersection
WHERE b.person_type IS NOT NULL
GROUP BY
    s.intersection,
    s.avg_latitude,
    s.avg_longitude,
    s.total_kabco,
    s.rank,
    b.person_type
ORDER BY s.rank, person_type_count DESC;

,intersection,avg_latitude,avg_longitude,total_kabco,rank,person_type,person_type_count
0,HOWARD ST & TITANIC AVE,31.853444,-106.433330,152154100.0,1,2 - PASSENGER/OCCUPANT,9
1,HOWARD ST & TITANIC AVE,31.853444,-106.433330,152154100.0,1,1 - DRIVER,5
2,HOWARD ST & TITANIC AVE,31.853444,-106.433330,152154100.0,1,4 - PEDESTRIAN,1
3,PEBBLE HILLS BLVD & TIERRA BLANDA DR,31.777093,-106.260784,68747500.0,2,1 - DRIVER,4
4,PEBBLE HILLS BLVD & TIERRA BLANDA DR,31.777093,-106.260784,68747500.0,2,2 - PASSENGER/OCCUPANT,1
...,...,...,...,...,...,...,...
3105,CASTNER DR & DIESEL DR,31.722412,-106.313484,0.0,834,1 - DRIVER,1
3106,COROZAL DR & LARIAT ST,31.751819,-106.351655,0.0,834,1 - DRIVER,1
3107,LA CADENA DR & PALMARY DR,31.846535,-106.546120,0.0,834,1 - DRIVER,1
3108,BLACKWOOD AVE & PARKWOOD ST,31.784765,-106.342370,0.0,834,2 - PASSENGER/OCCUPANT,1


In [32]:
%%sql
WITH base AS (
    SELECT
        crash_id,

        CONCAT(
            LEAST(street_name, intersecting_street_name),
            ' & ',
            GREATEST(street_name, intersecting_street_name)
        ) AS intersection,

        latitude,
        longitude,
        "Collision KABCO USD" AS kabco,

        contributing_factor_1,
        contributing_factor_2,
        contributing_factor_3,
        possible_contributing_factor_1,
        possible_contributing_factor_2,

        person_type,
        manner_of_collision

    FROM df
    WHERE street_name IS NOT NULL
      AND intersecting_street_name IS NOT NULL
      AND latitude IS NOT NULL
      AND longitude IS NOT NULL
),

intersection_summary AS (
    SELECT
        intersection,
        AVG(latitude) AS avg_latitude,
        AVG(longitude) AS avg_longitude,
        SUM(kabco) AS total_kabco,
        RANK() OVER (ORDER BY SUM(kabco) DESC) AS rank
    FROM base
    GROUP BY intersection
)SELECT
    s.intersection,
    s.avg_latitude,
    s.avg_longitude,
    s.total_kabco,
    s.rank,

    b.manner_of_collision,
    COUNT(*) AS collision_count

FROM intersection_summary s
LEFT JOIN base b
    ON s.intersection = b.intersection
WHERE b.manner_of_collision IS NOT NULL
GROUP BY
    s.intersection,
    s.avg_latitude,
    s.avg_longitude,
    s.total_kabco,
    s.rank,
    b.manner_of_collision
ORDER BY s.rank, collision_count DESC;

,intersection,avg_latitude,avg_longitude,total_kabco,rank,manner_of_collision,collision_count
0,HOWARD ST & TITANIC AVE,31.853444,-106.433330,152154100.0,1,ANGLE - BOTH GOING STRAIGHT,13
1,HOWARD ST & TITANIC AVE,31.853444,-106.433330,152154100.0,1,ONE MOTOR VEHICLE - GOING STRAIGHT,2
2,PEBBLE HILLS BLVD & TIERRA BLANDA DR,31.777093,-106.260784,68747500.0,2,OPPOSITE DIRECTION - ONE STRAIGHT-ONE LEFT TURN,5
3,DIANA DR & TETONS DR,31.858941,-106.423022,43370900.0,3,ANGLE - BOTH GOING STRAIGHT,11
4,DIANA DR & TETONS DR,31.858941,-106.423022,43370900.0,3,SAME DIRECTION - ONE STRAIGHT-ONE STOPPED,2
...,...,...,...,...,...,...,...
3011,NOLAN RICHARDSON DR & TOWER HILL DR,31.783195,-106.276250,0.0,834,ONE MOTOR VEHICLE - GOING STRAIGHT,1
3012,PARTELLO ST & PORTER AVE,31.804915,-106.446230,0.0,834,ONE MOTOR VEHICLE - GOING STRAIGHT,1
3013,OAXACA CT & VILLA MADERO DR,31.702027,-106.313971,0.0,834,ONE MOTOR VEHICLE - BACKING,1
3014,LOMA VERDE DR & LOMA VERDE DR,31.713940,-106.283300,0.0,834,ONE MOTOR VEHICLE - GOING STRAIGHT,1


In [30]:
contrib_pivot = contributing_factors_view.pivot_table(
    index=["intersection", "avg_latitude", "avg_longitude", "total_kabco", "rank"],
    columns="factor",
    values="factor_count",
    fill_value=0
).reset_index()

contrib_pivot

factor,intersection,avg_latitude,avg_longitude,total_kabco,rank,1 - ANIMAL ON ROAD - DOMESTIC,14 - DISABLED IN TRAFFIC LANE,15 - DISREGARD STOP AND GO SIGNAL,16 - DISREGARD STOP SIGN OR LIGHT,17 - DISREGARD TURN MARKS AT INTERSECTION,...,69 - WRONG SIDE - APPROACH OR INTERSECTION,71 - WRONG WAY - ONE WAY ROAD,73 - ROAD RAGE,74 - CELL/MOBILE DEVICE USE - TALKING,75 - CELL/MOBILE DEVICE USE - TEXTING,76 - CELL/MOBILE DEVICE USE - OTHER,77 - CELL/MOBILE DEVICE USE - UNKNOWN,78 - FAILED TO SLOW OR MOVE OVER FOR VEHICLES DISPLAYING EMERGENCY LIGHTS,98 - OTHER (EXPLAIN IN NARRATIVE),No Data
0,3RD AVE & CAMPBELL ST,31.755445,-106.482750,909600.0,271,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,25.0
1,4TH AVE & S CAMPBELL ST,31.754595,-106.482470,0.0,834,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,18.0
2,AARON ST & SEAN HAGGERTY DR,31.928355,-106.399910,303200.0,666,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
3,ABRIL DR & MONTWOOD DR,31.771457,-106.329164,606400.0,407,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0
4,ACER AVE & MCRAE BLVD,31.766414,-106.355860,0.0,834,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,16.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1918,WESTSIDE DR & WILLOW RIVER DR,31.880947,-106.625850,0.0,834,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,20.0
1919,WHITETAIL DR & WHITETAIL DR,31.909404,-106.398080,0.0,834,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
1920,WILLIAMETTE AVE & YERMOLAND DR,31.739003,-106.344475,1061200.0,252,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,23.0
1921,WILLOW ST & WYOMING AVE,31.775365,-106.470250,0.0,834,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,11.0


In [34]:
person_pivot = person_type_view.pivot_table(
    index=["intersection", "rank"],
    columns="person_type",
    values="person_type_count",
    fill_value=0
).reset_index()

person_pivot

person_type,intersection,rank,1 - DRIVER,2 - PASSENGER/OCCUPANT,3 - PEDALCYCLIST,4 - PEDESTRIAN,5 - DRIVER OF MOTORCYCLE TYPE VEHICLE,6 - PASSENGER/OCCUPANT ON MOTORCYCLE TYPE VEHICLE,98 - OTHER (EXPLAIN IN NARRATIVE),99 - UNKNOWN
0,3RD AVE & CAMPBELL ST,271,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,4TH AVE & S CAMPBELL ST,834,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0
2,AARON ST & SEAN HAGGERTY DR,666,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,ABRIL DR & MONTWOOD DR,407,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,ACER AVE & MCRAE BLVD,834,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
1918,WESTSIDE DR & WILLOW RIVER DR,834,2.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0
1919,WHITETAIL DR & WHITETAIL DR,834,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1920,WILLIAMETTE AVE & YERMOLAND DR,252,4.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1921,WILLOW ST & WYOMING AVE,834,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [33]:
collision_pivot = collisions_view.pivot_table(
    index=["intersection", "rank"],
    columns="manner_of_collision",
    values="collision_count",
    fill_value=0
).reset_index()

contrib_pivot

factor,intersection,avg_latitude,avg_longitude,total_kabco,rank,1 - ANIMAL ON ROAD - DOMESTIC,14 - DISABLED IN TRAFFIC LANE,15 - DISREGARD STOP AND GO SIGNAL,16 - DISREGARD STOP SIGN OR LIGHT,17 - DISREGARD TURN MARKS AT INTERSECTION,...,69 - WRONG SIDE - APPROACH OR INTERSECTION,71 - WRONG WAY - ONE WAY ROAD,73 - ROAD RAGE,74 - CELL/MOBILE DEVICE USE - TALKING,75 - CELL/MOBILE DEVICE USE - TEXTING,76 - CELL/MOBILE DEVICE USE - OTHER,77 - CELL/MOBILE DEVICE USE - UNKNOWN,78 - FAILED TO SLOW OR MOVE OVER FOR VEHICLES DISPLAYING EMERGENCY LIGHTS,98 - OTHER (EXPLAIN IN NARRATIVE),No Data
0,3RD AVE & CAMPBELL ST,31.755445,-106.482750,909600.0,271,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,25.0
1,4TH AVE & S CAMPBELL ST,31.754595,-106.482470,0.0,834,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,18.0
2,AARON ST & SEAN HAGGERTY DR,31.928355,-106.399910,303200.0,666,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
3,ABRIL DR & MONTWOOD DR,31.771457,-106.329164,606400.0,407,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0
4,ACER AVE & MCRAE BLVD,31.766414,-106.355860,0.0,834,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,16.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1918,WESTSIDE DR & WILLOW RIVER DR,31.880947,-106.625850,0.0,834,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,20.0
1919,WHITETAIL DR & WHITETAIL DR,31.909404,-106.398080,0.0,834,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
1920,WILLIAMETTE AVE & YERMOLAND DR,31.739003,-106.344475,1061200.0,252,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,23.0
1921,WILLOW ST & WYOMING AVE,31.775365,-106.470250,0.0,834,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,11.0


In [36]:
final = contrib_pivot.merge(person_pivot, on=["intersection", "rank"], how="left") \
                     .merge(collision_pivot, on=["intersection", "rank"], how="left")

# Sort by rank (ascending = highest severity first since rank 1 is highest)
final = final.sort_values("rank")

final

,intersection,avg_latitude,avg_longitude,total_kabco,rank,1 - ANIMAL ON ROAD - DOMESTIC,14 - DISABLED IN TRAFFIC LANE,15 - DISREGARD STOP AND GO SIGNAL,16 - DISREGARD STOP SIGN OR LIGHT,17 - DISREGARD TURN MARKS AT INTERSECTION,...,OTHER,SAME DIRECTION - BOTH GOING STRAIGHT-REAR END,SAME DIRECTION - BOTH GOING STRAIGHT-SIDESWIPE,SAME DIRECTION - BOTH LEFT TURN,SAME DIRECTION - BOTH RIGHT TURN,SAME DIRECTION - ONE LEFT TURN-ONE STOPPED,SAME DIRECTION - ONE RIGHT TURN-ONE STOPPED,SAME DIRECTION - ONE STRAIGHT-ONE LEFT TURN,SAME DIRECTION - ONE STRAIGHT-ONE RIGHT TURN,SAME DIRECTION - ONE STRAIGHT-ONE STOPPED
1093,HOWARD ST & TITANIC AVE,31.853444,-106.433330,152154100.0,1,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1630,PEBBLE HILLS BLVD & TIERRA BLANDA DR,31.777093,-106.260784,68747500.0,2,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
562,DIANA DR & TETONS DR,31.858941,-106.423022,43370900.0,3,0.0,0.0,1.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
1503,N PIEDRAS ST & N PIEDRAS ST,31.813975,-106.461320,41248500.0,4,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
833,ENCHANTED BREEZE DR & ENCHANTED PATH DR,31.918735,-106.572240,13749500.0,5,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33,ALABAMA ST & LEBANON AVE,31.794105,-106.465950,0.0,834,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
32,ALABAMA ST & IDALIA AVE,31.804055,-106.465890,0.0,834,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31,ALABAMA ST & HUECO VISTA WAY,31.834665,-106.456190,0.0,834,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
27,AIRWAY BLVD & W GATEWAY BLVD,31.778889,-106.392505,0.0,834,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,5.0,0.0,0.0,0.0,4.0,0.0,4.0


In [153]:
%%sql
WITH non_zero AS (
    SELECT longitude, latitude, "Collision KABCO USD", crash_id, crash_month, crash_year, light_condition,
                      manner_of_collision, speed_limit, contributing_factor_1, person_type
    FROM df
    WHERE "Collision KABCO USD" != 0
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (ORDER BY "Collision KABCO USD" DESC) AS rn,
           COUNT(*) OVER () AS total_count
    FROM non_zero
)
SELECT *
FROM ranked

,longitude,latitude,Collision KABCO USD,crash_id,crash_month,crash_year,light_condition,manner_of_collision,speed_limit,contributing_factor_1,person_type,rn,total_count
0,-106.423010,31.858945,13749500,17569235,1,2020,"2 - DARK, NOT LIGHTED",ANGLE - BOTH GOING STRAIGHT,35,20 - DRIVER INATTENTION,1 - DRIVER,1,3735
1,-106.423010,31.858945,13749500,17569235,1,2020,"2 - DARK, NOT LIGHTED",ANGLE - BOTH GOING STRAIGHT,35,20 - DRIVER INATTENTION,2 - PASSENGER/OCCUPANT,2,3735
2,-106.423010,31.858945,13749500,17569235,1,2020,"2 - DARK, NOT LIGHTED",ANGLE - BOTH GOING STRAIGHT,35,No Data,1 - DRIVER,3,3735
3,-106.461320,31.813975,13749500,17553630,2,2020,"2 - DARK, NOT LIGHTED",ANGLE - BOTH GOING STRAIGHT,30,22 - FAILED TO CONTROL SPEED,1 - DRIVER,4,3735
4,-106.461320,31.813975,13749500,17553630,2,2020,"2 - DARK, NOT LIGHTED",ANGLE - BOTH GOING STRAIGHT,30,No Data,1 - DRIVER,5,3735
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3730,-106.484043,31.756174,151600,20067749,3,2024,"2 - DARK, NOT LIGHTED",OPPOSITE DIRECTION - ONE BACKING-ONE STOPPED,35,3 - BACKED WITHOUT SAFETY,1 - DRIVER,3731,3735
3731,-106.433632,31.904407,151600,19716072,8,2023,1 - DAYLIGHT,SAME DIRECTION - BOTH GOING STRAIGHT-REAR END,35,20 - DRIVER INATTENTION,1 - DRIVER,3732,3735
3732,-106.365252,31.762529,151600,19699383,8,2023,1 - DAYLIGHT,SAME DIRECTION - BOTH GOING STRAIGHT-SIDESWIPE,35,No Data,2 - PASSENGER/OCCUPANT,3733,3735
3733,-106.485030,31.767015,151600,19614455,6,2023,1 - DAYLIGHT,ANGLE - BOTH GOING STRAIGHT,35,No Data,2 - PASSENGER/OCCUPANT,3734,3735


In [138]:
discrete_scale = [
    [0.0,   "#0b3c5d"],
    [0.1667,"#0b3c5d"],

    [0.1667,"#3a7ca5"],
    [0.3333,"#3a7ca5"],

    [0.3333,"#6c8ebf"],
    [0.5,   "#6c8ebf"],

    [0.5,   "#e28743"],
    [0.6667,"#e28743"],

    [0.6667,"#d66a1f"],
    [0.8333,"#d66a1f"],

    [0.8333,"#b85c0a"],
    [1.0,   "#b85c0a"]
]

# 10 evenly spaced tick values from 0 to pre_max
tick_vals = np.linspace(0, 6000000, 6)
tick_text = [f"{int(v):,}" for v in tick_vals]  # formatted with commas

ColorVariable = discrete_scale
mapStyle = "carto-positron"

In [139]:
# Normalize total_kabco to range [0.2, 1] (avoid fully invisible points)
min_opacity = 0.2
kabco = final['total_kabco']

opacity = min_opacity + (kabco - kabco.min()) / (kabco.max() - kabco.min()) * (1 - min_opacity)

In [163]:
ColorVariable = discrete_scale
mapStyle = "open-street-map"

pre_max = final['total_kabco'].max()

fig = go.Figure()

# ---- Pre LPI series (with colorbar) ----
fig.add_trace(go.Scattermapbox(
    lat=final['avg_latitude'],
    lon=final['avg_longitude'],
    mode='markers',
    marker=go.scattermapbox.Marker(
        size=8,
        color=final['total_kabco'],
        colorscale=ColorVariable,
        cmin=0,
        cmax=pre_max,
        opacity=opacity,   # 👈 key addition
        showscale=True,
        colorbar=dict(
            title='KABCO sum (USD)',
            orientation='h',
            x=0.5,
            y=-0.1,
            len=0.9
        )
    ),
    text=final['intersection'],
    hovertemplate=(
        "<b>%{text}</b><br>"
        "Intersection"
    ),
    customdata=final[['total_kabco']],
    name='Ranked Intersections'
))

# ---- Pre LPI series (with colorbar and enhanced hover) ----
fig.add_trace(go.Scattermapbox(
    lat=collisions_non_zero_KABCO['latitude'],
    lon=collisions_non_zero_KABCO['longitude'],
    mode='markers',
    marker=go.scattermapbox.Marker(
        size=np.log(collisions_non_zero_KABCO['Collision KABCO USD']),
        color=np.log(collisions_non_zero_KABCO['Collision KABCO USD']),
        colorscale=ColorVariable,
        showscale=True,
        colorbar=dict(
            title='KABCO sum (USD)',
            orientation='h',
            x=0.5,
            y=-0.1,
            len=0.9
        )
    ),
    customdata=collisions_non_zero_KABCO[['crash_id', 'crash_month', 'crash_year',
                                          'light_condition', 'manner_of_collision',
                                          'speed_limit', 'contributing_factor_1',
                                          'person_type', 'Collision KABCO USD']],
    hovertemplate=(
        "<b>%{text}</b><br>"  # intersection
        "Crash ID: %{customdata[0]}<br>"
        "Month: %{customdata[1]}<br>"
        "Year: %{customdata[2]}<br>"
        "Light Condition: %{customdata[3]}<br>"
        "Manner of Collision: %{customdata[4]}<br>"
        "Speed Limit: %{customdata[5]} mph<br>"
        "Contributing Factor 1: %{customdata[6]}<br>"
        "Person Type: %{customdata[7]}<br>"
        "<b>KABCO USD: %{customdata[8]:,.0f}</b><br>"
        "<extra></extra>"
    ),
    name='Ranked Intersections',
    showlegend=True
))

# 2️⃣ Overlay weighted density directly on Mapbox
fig.add_trace(go.Densitymapbox(
    lat=collisions_non_zero_KABCO["latitude"],
    lon=collisions_non_zero_KABCO["longitude"],
    z=np.log(collisions_non_zero_KABCO["Collision KABCO USD"]),  # weights
    radius=26,        # controls size of smoothing
    colorscale="Viridis",
    opacity=0.75,      # semi-transparent
    showscale=True,
    name="Density Heatmap",   # 👈 add this
    colorbar=dict(
        title="Log(KABCO) Weighted Density",
        orientation="h",
        y=-0.15,
        x=0.5,
        len=0.9,
        xanchor="center",
        yanchor="top",
        thickness=15
    ),
    showlegend=True
))

# Add north arrow (text "N") in top-right corner
fig.add_annotation(
    x=0.95,  # relative position (0 = left, 1 = right)
    y=0.95,  # relative position (0 = bottom, 1 = top)
    xref="paper",
    yref="paper",
    text="⬆ N",
    showarrow=False,
    font=dict(size=30, color="black"),
    align="center"
)

# ---- Layout ----
fig.update_layout(
    mapbox_style=mapStyle,
    mapbox_zoom=10,
    mapbox_center={"lat": final["avg_latitude"].mean(), "lon": final["avg_longitude"].mean()},
    mapbox=dict(
        bearing=0,
        pitch=0
    ),
    title=dict(
        text="<b>Collisions at Intersections</b>",
        font=dict(size=20),
        y=0.99,                  # slightly lower than the top (default is 1.0)
        pad=dict(t=1, b=0),     # add 20px padding on top
    ),
    legend=dict(title="Layers"),
    hovermode="closest",
)

# ---- Show map ----
fig.show(renderer="browser", config={'scrollZoom': True})

In [46]:
%%sql
SELECT *
FROM final
LIMIT 10

,intersection,avg_latitude,avg_longitude,total_kabco,rank,1 - ANIMAL ON ROAD - DOMESTIC,14 - DISABLED IN TRAFFIC LANE,15 - DISREGARD STOP AND GO SIGNAL,16 - DISREGARD STOP SIGN OR LIGHT,17 - DISREGARD TURN MARKS AT INTERSECTION,...,OTHER,SAME DIRECTION - BOTH GOING STRAIGHT-REAR END,SAME DIRECTION - BOTH GOING STRAIGHT-SIDESWIPE,SAME DIRECTION - BOTH LEFT TURN,SAME DIRECTION - BOTH RIGHT TURN,SAME DIRECTION - ONE LEFT TURN-ONE STOPPED,SAME DIRECTION - ONE RIGHT TURN-ONE STOPPED,SAME DIRECTION - ONE STRAIGHT-ONE LEFT TURN,SAME DIRECTION - ONE STRAIGHT-ONE RIGHT TURN,SAME DIRECTION - ONE STRAIGHT-ONE STOPPED
0,HOWARD ST & TITANIC AVE,31.853444,-106.433330,152154100.0,1,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,PEBBLE HILLS BLVD & TIERRA BLANDA DR,31.777093,-106.260784,68747500.0,2,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,DIANA DR & TETONS DR,31.858941,-106.423022,43370900.0,3,0.0,0.0,1.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
3,N PIEDRAS ST & N PIEDRAS ST,31.813975,-106.461320,41248500.0,4,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,ENCHANTED BREEZE DR & ENCHANTED PATH DR,31.918735,-106.572240,13749500.0,5,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,BOB MITCHELL DR & SAUL KLEINFELD DR,31.761330,-106.280814,11459200.0,6,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,2.0,3.0
6,EDGEMERE BLVD & N LEE TREVINO DR,31.791555,-106.323334,11459200.0,6,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,EDGEMERE BLVD & NOLAN RICHARDSON DR,31.792262,-106.276345,11020000.0,8,0.0,1.0,1.0,1.0,0.0,...,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.0
8,MCCOMBS ST & WOODROW BEAN TRANSMOUNTAIN DR,31.898953,-106.406787,10952000.0,9,0.0,0.0,3.0,0.0,0.0,...,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0
9,ALABAMA ST & RICHMOND AVE,31.795915,-106.465940,9739200.0,10,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [50]:
%%sql
SELECT *
FROM contributing_factors_view
WHERE rank <= 10
AND factor != 'No Data'

,intersection,avg_latitude,avg_longitude,total_kabco,rank,factor,factor_count
0,HOWARD ST & TITANIC AVE,31.853444,-106.433330,152154100.0,1,20 - DRIVER INATTENTION,8
1,HOWARD ST & TITANIC AVE,31.853444,-106.433330,152154100.0,1,45 - HAD BEEN DRINKING,7
2,HOWARD ST & TITANIC AVE,31.853444,-106.433330,152154100.0,1,60 - UNSAFE SPEED,4
3,HOWARD ST & TITANIC AVE,31.853444,-106.433330,152154100.0,1,16 - DISREGARD STOP SIGN OR LIGHT,1
4,HOWARD ST & TITANIC AVE,31.853444,-106.433330,152154100.0,1,19 - DISTRACTION IN VEHICLE,1
5,HOWARD ST & TITANIC AVE,31.853444,-106.433330,152154100.0,1,36 - FAILED TO YIELD RIGHT OF WAY - TO PEDESTRIAN,1
6,PEBBLE HILLS BLVD & TIERRA BLANDA DR,31.777093,-106.260784,68747500.0,2,37 - FAILED TO YIELD RIGHT OF WAY - TURNING LEFT,2
7,PEBBLE HILLS BLVD & TIERRA BLANDA DR,31.777093,-106.260784,68747500.0,2,20 - DRIVER INATTENTION,1
8,PEBBLE HILLS BLVD & TIERRA BLANDA DR,31.777093,-106.260784,68747500.0,2,98 - OTHER (EXPLAIN IN NARRATIVE),1
9,PEBBLE HILLS BLVD & TIERRA BLANDA DR,31.777093,-106.260784,68747500.0,2,22 - FAILED TO CONTROL SPEED,1


In [51]:
%%sql
SELECT *
FROM person_type_view
WHERE rank <= 10

,intersection,avg_latitude,avg_longitude,total_kabco,rank,person_type,person_type_count
0,HOWARD ST & TITANIC AVE,31.853444,-106.433330,152154100.0,1,2 - PASSENGER/OCCUPANT,9
1,HOWARD ST & TITANIC AVE,31.853444,-106.433330,152154100.0,1,1 - DRIVER,5
2,HOWARD ST & TITANIC AVE,31.853444,-106.433330,152154100.0,1,4 - PEDESTRIAN,1
3,PEBBLE HILLS BLVD & TIERRA BLANDA DR,31.777093,-106.260784,68747500.0,2,1 - DRIVER,4
4,PEBBLE HILLS BLVD & TIERRA BLANDA DR,31.777093,-106.260784,68747500.0,2,2 - PASSENGER/OCCUPANT,1
5,DIANA DR & TETONS DR,31.858941,-106.423022,43370900.0,3,1 - DRIVER,9
6,DIANA DR & TETONS DR,31.858941,-106.423022,43370900.0,3,2 - PASSENGER/OCCUPANT,5
7,N PIEDRAS ST & N PIEDRAS ST,31.813975,-106.461320,41248500.0,4,1 - DRIVER,2
8,N PIEDRAS ST & N PIEDRAS ST,31.813975,-106.461320,41248500.0,4,2 - PASSENGER/OCCUPANT,1
9,ENCHANTED BREEZE DR & ENCHANTED PATH DR,31.918735,-106.572240,13749500.0,5,5 - DRIVER OF MOTORCYCLE TYPE VEHICLE,1


In [52]:
%%sql
SELECT *
FROM collisions_view
WHERE rank <= 10

,intersection,avg_latitude,avg_longitude,total_kabco,rank,manner_of_collision,collision_count
0,HOWARD ST & TITANIC AVE,31.853444,-106.433330,152154100.0,1,ANGLE - BOTH GOING STRAIGHT,13
1,HOWARD ST & TITANIC AVE,31.853444,-106.433330,152154100.0,1,ONE MOTOR VEHICLE - GOING STRAIGHT,2
2,PEBBLE HILLS BLVD & TIERRA BLANDA DR,31.777093,-106.260784,68747500.0,2,OPPOSITE DIRECTION - ONE STRAIGHT-ONE LEFT TURN,5
3,DIANA DR & TETONS DR,31.858941,-106.423022,43370900.0,3,ANGLE - BOTH GOING STRAIGHT,11
4,DIANA DR & TETONS DR,31.858941,-106.423022,43370900.0,3,SAME DIRECTION - ONE STRAIGHT-ONE STOPPED,2
5,DIANA DR & TETONS DR,31.858941,-106.423022,43370900.0,3,ONE MOTOR VEHICLE - GOING STRAIGHT,1
6,N PIEDRAS ST & N PIEDRAS ST,31.813975,-106.461320,41248500.0,4,ANGLE - BOTH GOING STRAIGHT,3
7,ENCHANTED BREEZE DR & ENCHANTED PATH DR,31.918735,-106.572240,13749500.0,5,ONE MOTOR VEHICLE - GOING STRAIGHT,1
8,EDGEMERE BLVD & N LEE TREVINO DR,31.791555,-106.323334,11459200.0,6,ANGLE - BOTH GOING STRAIGHT,11
9,BOB MITCHELL DR & SAUL KLEINFELD DR,31.761330,-106.280814,11459200.0,6,ANGLE - ONE STRAIGHT-ONE LEFT TURN,8
